# Prefix Sums & Difference Arrays

Two mirror-image pre-computation patterns. A **prefix sum** answers any **range-sum query** in O(1) after one O(n) pass. A **difference array** applies any number of **range updates** in O(1) each, then materializes in one pass. They're inverses — prefix-summing a difference array undoes the differencing — and together they turn repeated range work from O(n) per operation into O(1).

> signal → template → worked examples → toolkit

**Contents**

1. **The signal**
2. **Prefix sums** — O(1) range queries
3. **Prefix sums + a hashmap** — subarray sum = k
4. **Difference arrays** — O(1) range updates
5. **2D prefix sums**
6. **When to use & toolkit**
7. **Complexity cheat-sheet**

## 1. The signal — prefix sums vs difference arrays

Two complementary situations, both on a **fixed-size** array with **many** operations:

- **Many range-*sum* queries** ("sum of `a[l..r]`", asked repeatedly) → build a **prefix-sum** array once, answer each query by subtraction in O(1).
- **Many range-*updates*** ("add v to `a[l..r]`", applied repeatedly, result read at the end) → record the deltas in a **difference array** in O(1) each, then prefix-sum it once for the final array.

They're **inverses**: differencing then prefix-summing recovers the original, and vice versa. Recognizing *which direction* you need is the whole trick.

## 2. Prefix sums — O(1) range queries

`prefix[i]` holds the sum of the first `i` elements (with `prefix[0] = 0`). Then the sum of any inclusive range `[l, r]` is just **`prefix[r+1] - prefix[l]`** — two lookups, no loop. Building the array is one O(n) pass; every query after that is O(1).

In [1]:
def build_prefix(a):
    prefix = [0] * (len(a) + 1)          # prefix[0] = 0; prefix[i] = sum(a[:i])
    for i, x in enumerate(a):
        prefix[i + 1] = prefix[i] + x
    return prefix

def range_sum(prefix, l, r):             # inclusive [l, r]
    return prefix[r + 1] - prefix[l]

a = [3, 1, 4, 1, 5, 9, 2, 6]
prefix = build_prefix(a)
print("sum of a[2..5]:", range_sum(prefix, 2, 5))     # 4+1+5+9 = 19
print("sum of a[0..7]:", range_sum(prefix, 0, 7))

# the stdlib builds the running totals for you:
from itertools import accumulate
print("accumulate    :", list(accumulate(a)))         # [3, 4, 8, 9, 14, 23, 25, 31]


sum of a[2..5]: 19
sum of a[0..7]: 31
accumulate    : [3, 4, 8, 9, 14, 23, 25, 31]


## 3. Prefix sums + a hashmap — subarray sum = k

The same idea unlocks a problem **sliding window can't** handle: count subarrays summing to exactly `k` when values may be **negative** (so window sums aren't monotonic — the caveat from the sliding-window notebook). A subarray `a[i..j]` sums to `k` iff `prefix[j+1] - prefix[i] == k`, i.e. `prefix[i] == prefix[j+1] - k`. So scan left to right and, for each running prefix, count how many earlier prefixes equal `running - k` — one O(n) pass with a hashmap of prefix counts:

In [2]:
from collections import defaultdict

def subarray_sum_count(a, k):
    seen = defaultdict(int)
    seen[0] = 1                          # the empty prefix (sum 0) has occurred once
    running = count = 0
    for x in a:
        running += x
        count += seen[running - k]       # earlier prefixes that close a window summing to k
        seen[running] += 1
    return count

print("[1,2,3], k=3                  :", subarray_sum_count([1, 2, 3], 3))
print("[1,-1,5,-2,3], k=3 (negatives):", subarray_sum_count([1, -1, 5, -2, 3], 3))


[1,2,3], k=3                  : 2
[1,-1,5,-2,3], k=3 (negatives): 3


## 4. Difference arrays — O(1) range updates

The mirror image. To add `v` across `a[l..r]` you'd normally touch r−l+1 cells; with a **difference array** you touch just **two**: `diff[l] += v` and `diff[r+1] -= v`. After all updates, a single prefix-sum pass over `diff` reconstructs the final array — so q range updates cost **O(q + n)** instead of O(q·n):

In [3]:
def apply_range_updates(n, updates):
    diff = [0] * (n + 1)                 # one extra slot for the r+1 marker
    for l, r, v in updates:              # add v to every index in [l, r]
        diff[l] += v
        diff[r + 1] -= v
    result, running = [], 0
    for i in range(n):                   # prefix-sum the deltas to materialize
        running += diff[i]
        result.append(running)
    return result

# add 3 to [0,2], then 2 to [1,3], then -1 to [2,4]:
print("after range updates:", apply_range_updates(5, [(0, 2, 3), (1, 3, 2), (2, 4, -1)]))


after range updates: [3, 5, 4, 1, -1]


This is the engine behind problems like "car pooling," "corporate flight bookings," and any "range increment" task — wherever updates batch up and you read the result only once at the end.

## 5. 2D prefix sums

Prefix sums extend to a grid: `P[i][j]` = the sum of the whole rectangle above-and-left of `(i, j)`. Building it uses inclusion-exclusion (add the two overlapping rectangles, subtract the doubly-counted corner), and **any submatrix sum** then comes out in O(1) with the same trick:

In [4]:
def build_2d_prefix(grid):
    R, C = len(grid), len(grid[0])
    P = [[0] * (C + 1) for _ in range(R + 1)]
    for i in range(R):
        for j in range(C):
            P[i + 1][j + 1] = grid[i][j] + P[i][j + 1] + P[i + 1][j] - P[i][j]
    return P

def region_sum(P, r1, c1, r2, c2):       # inclusive corners
    return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]

grid = [[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]]
P = build_2d_prefix(grid)
print("sum of submatrix (1,1)-(2,2):", region_sum(P, 1, 1, 2, 2))   # 5+6+8+9 = 28


sum of submatrix (1,1)-(2,2): 28


## 6. When to use & toolkit

| You have… | Use | After the O(n) prep |
|---|---|---|
| Many range-**sum** queries, fixed array | prefix sums | O(1) per query |
| Subarray-sum / count with **negatives** | prefix sums + hashmap | O(n) one pass |
| Many range-**updates**, read once at the end | difference array | O(1) per update |
| Submatrix sums on a grid | 2D prefix sums | O(1) per query |

**Python toolkit:** `itertools.accumulate(a)` builds the running sums (and with a `func=` argument, running min/max/products); `accumulate` over a difference array *is* the materialize step. numpy's `cumsum` does the same, vectorized, for large numeric arrays.

**Watch for:** prefix sums assume the array **doesn't change** between queries. If it does, you need a Fenwick / Binary-Indexed Tree or a segment tree (O(log n) per update *and* query) — see the **Fenwick & segment trees** notebook. Prefix sums are the static-array special case of those.

## 7. Complexity cheat-sheet

| Technique | Build | Query | Update |
|---|:---:|:---:|:---:|
| Prefix sum (1D) | O(n) | O(1) range sum | O(n) (rebuild) |
| Difference array | O(n + q) total | O(n) read-out | O(1) range add |
| 2D prefix sum | O(R·C) | O(1) submatrix | O(R·C) (rebuild) |
| Fenwick tree *(dynamic)* | O(n) | O(log n) | O(log n) |

<sub>Prefix sums and difference arrays are the cheap, **static** answer; reach for a Fenwick / segment tree only when the array is updated *and* queried interleaved.</sub>